# 🤖 Social Media Post AI Agent
### David Scott & Senthil Vel — Agentic AI & Prompt Engineering, Saint Joseph's University (Spring 2026)

---

## 🎯 Objective & The Problem

For those working at a marketing agency — or even volunteering as a marketing manager for an organization — a large portion of time is often spent creating social media posts that are frequent but relatively routine. These posts are important for keeping audiences engaged, but the time they take adds up quickly, especially across multiple clients or platforms. That leaves less time for the creative, high-impact work: complex campaigns, video content, strategy, and analytics.

This AI Agent Prototype was built to solve that problem. It creates and schedules on-brand social media posts for a specific company's product or event, and remembers everything it learned — so future visits skip straight to generating new posts in the same style without starting over.

**Why not just use a simple ChatGPT prompt?**
- No brand memory across sessions — starts from scratch every time
- No ability to search the web, read files, or fetch URLs for research
- No automated quality checks or guardrails on the output
- No multi-platform parallel generation
- No Google Calendar integration

---

## 🧠 Agentic Patterns Used

This project combines several agentic patterns:

1. **Sequential Workflow** — A clear step-by-step pipeline where each stage's output feeds the next: User Input → Receipt → Research → Style Guide → Image Selection → Post Generation → Audit → Save → Schedule.

2. **Human-in-the-Loop** — The user stays in control at key decision points: confirming the parsed receipt, approving fuzzy name matches ("Is 'Rita's' the same as 'Rita's Italian Ice'?"), selecting images, and confirming the schedule before it pushes to Google Calendar.

3. **ReAct (Reason + Act)** — The Research Agent runs a Thought → Action → Observation loop, deciding at each step whether to search the web, fetch a URL, read a document, or ask the user a clarifying question. Uses OpenAI function calling for reliable structured tool use.

4. **Parallelization** — Within post generation, `InstagramAgent` and `TwitterAgent` run simultaneously using `ThreadPoolExecutor`, each generating and scoring their own caption at the same time.

5. **Evaluator-Optimizer** — After image generation, an auditor evaluates the result and sends escalating correction prompts (concise → emotional urgency → nuclear) until the image passes or max attempts are reached. "No image is better than a bad image."

---

## 📁 Output Folder Structure (created automatically)

```
AI Storage/
  {Company}/
    {Company} Company Report.json
    Products/
      {Product} Product Report.json
    Style Guides/
      {Product} Style Guide.json
    Images/
      {Product}/
        photo.jpg               ← product photos
        references/             ← style reference screenshots
    Created Posts/
      {date} – Request {Product}/
        Post 1/
          caption.txt
          post_image.png
        Receipt of Creation.txt
    Log Files/
      {date} – {Company} – {Product} – Receipt.txt
```

---

**Getting Started:** Read the `README.md` first for project context, then see `Help Documents/INSTRUCTIONS.md` for full setup and `Help Documents/HELP_GUIDE.md` for a breakdown of every agent and function.


---
## 📦 Step 0: Imports & Setup

Loads all the agent modules used throughout this notebook. Each agent lives in its own Python file in the same folder — `agent_research.py`, `agent_base.py`, `agent_posts.py`, etc. — keeping responsibilities clearly separated and making each one easy to find and update independently.

In [ ]:
import os, json, re, datetime, time
from pathlib import Path

from agent_storage  import Storage
from agent_research import ResearchAgent
from agent_posts    import PLATFORM_AGENTS, build_context, parse_platforms
from agent_parallel import run_all_posts
from agent_schedule import ScheduleAgent
from agent_logger   import Logger
from agent_utils    import confirm, print_receipt

print('✅ Imports OK')

---
## 🔑 Step 1: API Keys

Loads API keys from a `.env` file using `python-dotenv`. Keys are never hardcoded in the notebook — the `.env` file is in `.gitignore` so it never ends up in the GitHub repo.

- `OPENAI_API_KEY` — used for all GPT and image generation calls
- `OPENAI_REVIEW_KEY` — same key, feeds the `gpt-4o-mini` A/B scorer and ReAct loop to keep costs down
- `GCAL_ID` / `GCAL_CREDENTIALS` — only needed for Google Calendar scheduling (optional)

In [ ]:
from dotenv import load_dotenv

# Loads all keys from .env automatically
load_dotenv()

OPENAI_API_KEY        = os.getenv('OPENAI_API_KEY')
OPENAI_REVIEW_KEY     = os.getenv('OPENAI_API_KEY')   # same key, cheaper model used internally
GOOGLE_CALENDAR_ID    = os.getenv('GCAL_ID', 'primary')
GCAL_CREDENTIALS_JSON = os.getenv('GCAL_CREDENTIALS', 'credentials.json')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not found. Did you fill in your .env file?')

print('✅ Keys loaded from .env')

---
## 📁 Step 2: Storage Setup

`Storage` is the file system manager for the entire project. It handles all folder creation, JSON reading and writing, and image management. No agent ever writes files directly — everything goes through `Storage`. This means the folder structure can be changed in one place without touching any agent code.

The `AI Storage/` folder is created automatically next to this notebook on first run.

In [ ]:
PROJECT_ROOT = Path('.')   # same folder as this notebook
storage = Storage(PROJECT_ROOT / 'AI Storage')
print(f'📂 AI Storage: {storage.root.resolve()}')

---
## 🗣️ Step 3: Parse User Request

The user writes a plain English request and `parse_request()` extracts structured fields using regex — platform, post count, company, product, month, schedule flag, and image mode. The output is called a **receipt**, which drives every step that follows.

Rather than asking the user to fill out a form, they describe what they want naturally and the system figures out the intent. This is the first prompt engineering decision in the project.

Image mode defaults are set per platform in `IMAGE_PLATFORM_DEFAULTS` — Instagram defaults to `Provided Images`, Twitter defaults to `No image`. These can be overridden in the request ("without image", "AI generated images") or corrected in Step 6.

In [ ]:
# ── Change this to your request ────────────────────────────────────────────
USER_REQUEST = "Create 1 Instagram and 2 Twitter posts for Rita's Kiwi Melon in June + Schedule. Make it an interactive post."

from agent_utils import parse_request
receipt = parse_request(USER_REQUEST)
print_receipt(receipt)

---
## 🔍 Step 4: Company & Product Lookup

**Agentic Pattern: ReAct + Function Calling**

This is where the Research Agent runs. It uses the **ReAct pattern** — a Thought → Action → Observation loop where GPT reasons about what it needs to know, picks a tool, gets a result, reasons about that result, and keeps going until it has enough to write a full report.

**Four tools available:** `web_search`, `fetch_url`, `read_document`, `ask`

**Two models used:** `gpt-4o-mini` handles tool routing in the loop (fast and cheap for simple decisions), then `gpt-4o` writes the final structured report (requires stronger reasoning).

**Fuzzy name matching** runs before research — so "Rita's" finds "Rita's Italian Ice" and loads the existing report instead of re-researching. Uses `SequenceMatcher` (ratio > 0.6) and substring containment, with a Human-in-the-Loop confirmation before acting on any match.

**Smart null field check after the loop:**
- **1 field missing** → asks you one targeted question for that specific field (e.g. "Do you know the price range for Kiwi Melon?")
- **2+ fields missing** → asks for a URL or file to run a second research pass
- **0 fields missing** → moves on automatically

**On return visits:** Both reports are already saved — this step loads them instantly and skips the ReAct loop entirely.

In [ ]:
researcher = ResearchAgent(
    storage=storage,
    openai_key=OPENAI_API_KEY,
)

# Optional: set these if you have a specific URL or uploaded document
# provided_url  = 'https://example.com/about'
# uploaded_file = 'my_product_brief.pdf'
provided_url  = None
uploaded_file = None


In [ ]:
company_report, product_report, resolved_company, resolved_product = researcher.resolve(
    company_name=receipt['company'],
    product_name=receipt['product'],
    provided_url=provided_url,
    uploaded_file=uploaded_file,
)

# Update receipt with the canonical names resolved by the agent
# (e.g. 'Rita\'s' becomes 'Rita\'s Water Ice' after fuzzy match / official name lookup)
receipt['company'] = resolved_company
receipt['product'] = resolved_product
print(f'  ✅ Receipt updated: company=[{resolved_company}], product=[{resolved_product}]')

---
## 📖 Step 5: Review Reports (Optional)

Gives you a chance to inspect the Company and Product reports before continuing. Reports are saved as JSON files in `AI Storage/` — you can also edit them directly in any text editor.

On return visits this step is typically skipped — the reports are already accurate from the first run.

In [ ]:
want_review = confirm('Would you like to review the Company or Product report before continuing? (y/N)')
if want_review:
    choice = input("Type 'company', 'product', or 'both': ").strip().lower()
    if choice in ('company', 'both'):
        print(json.dumps(company_report, indent=2))
    if choice in ('product', 'both'):
        print(json.dumps(product_report, indent=2))
    input('\nPress Enter when done reviewing...')

---
## ✅ Step 6: Confirm Receipt

**Agentic Pattern: Human-in-the-Loop**

Shows the parsed receipt and lets you correct any field before generation starts. This is important because the regex parser makes reasonable guesses — but you stay in control.

Type `modify <field> to <value>` to change anything (e.g. `modify images to AI Generated`). Press Enter or type `all good` to confirm.

Note: `num_posts` is locked when multiple platforms are specified — it's derived from the per-platform counts in your request.

In [ ]:
from agent_utils import interactive_receipt_editor
receipt = interactive_receipt_editor(receipt)
print_receipt(receipt)

---
## 🎨 Step 7: Style References

Add screenshots of posts you want the agent to imitate. These are passed to `gpt-4.1` during caption drafting so the agent can see the style, tone, and energy of real posts before writing.

References are stored in `AI Storage/{Company}/Images/{Product}/references/` and persist across sessions — add them once and they're available every time.

**Commands:** `addref <filepath>` to add | `rmref <number>` to remove | Enter to continue

If references change, you'll be asked whether to regenerate the style guide in the next step.

In [ ]:
ref_dir = storage.references_dir(receipt['company'], receipt['product'])
ref_dir.mkdir(parents=True, exist_ok=True)

# Snapshot BEFORE the loop so we can detect any changes after
refs_before_loop = set(r.name for r in storage.list_references(receipt['company'], receipt['product']))
existing_refs = storage.list_references(receipt['company'], receipt['product'])

print(f'\n🎨 Style references for [{receipt["product"]}]:')
if existing_refs:
    print(f'  Found {len(existing_refs)} reference(s):')
    for i, r in enumerate(existing_refs, 1):
        print(f'    {i}. {r.name}')
else:
    print('  No style references yet.')

print('\n  Commands: addref <filepath> | rmref <number> | Enter')
print('  Tip: pasting a file path directly also works as addref.')
print()

while True:
    cmd = input('  ref> ').strip().strip('"').strip("'")
    if not cmd:
        break

    parts  = cmd.split(None, 1)
    action = parts[0].lower()
    arg    = parts[1].strip() if len(parts) > 1 else ''

    # Auto-detect file path — treat as addref without typing the command
    is_path = ('\\' in cmd or '/' in cmd or
               cmd.lower().endswith(('.png', '.jpg', '.jpeg')))
    if is_path and action not in ('addref', 'rmref'):
        action = 'addref'
        arg    = cmd

    if action == 'addref' and arg:
        added = storage.add_reference_to_library(receipt['company'], receipt['product'], arg)
        if added:
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
    elif action == 'rmref' and arg.isdigit():
        idx = int(arg) - 1
        if 0 <= idx < len(existing_refs):
            existing_refs[idx].unlink()
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
            print('  🗑️  Removed.')
        else:
            print('  ❌ Invalid number.')
    else:
        print('  addref <filepath> | rmref <number> | Enter')

reference_images = storage.list_references(receipt['company'], receipt['product'])
print(f'\n✅ {len(reference_images)} style reference(s) loaded.')

In [ ]:
# ── Detect reference changes and ask about style guide update ────────────
# Compares references before and after Step 7.
# If changed and a style guide already exists, offers 3 options.
# Future improvement: replace numbered menu with GPT classification
# Intent classification — natural language in, structured label out.

refs_after   = set(r.name for r in storage.list_references(receipt['company'], receipt['product']))
refs_changed = refs_after != refs_before_loop  # compare against snapshot taken before Step 7
has_existing_guide = storage.style_guide_exists(receipt['company'], receipt['product'])

update_mode = 'leave'  # default

if refs_changed and has_existing_guide:
    print('\n🎨 References changed and a style guide already exists.')
    print('  What would you like to do?')
    print('  1. Leave as-is (references still influence posts directly)')
    print('  2. Touch up   (lightly blend new references into existing style)')
    print('  3. Regenerate (full rewrite using all current references)')
    choice = input('  Choice (1/2/3, default 1): ').strip()
    if choice == '2':
        update_mode = 'touchup'
    elif choice == '3':
        update_mode = 'regenerate'
    else:
        update_mode = 'leave'
    print(f'  ✅ Style guide update mode: {update_mode}')
else:
    print('ℹ️  No reference changes detected — style guide will load as-is if it exists.')

---
## 🎨 Step 7.5: Generate Style Guide

The style guide captures brand vibe, tone, color palette, emoji usage, and visual themes. It is generated **after** style references are added (Step 7) so `gpt-4.1` can analyze the reference screenshots and factor their visual style directly into the guide.

The style guide is passed to every agent during post generation — it's what ensures posts feel consistent across platforms and over time.

**On return visits:** Loads the existing guide instantly. If you added new references in Step 7, you'll be asked whether to update it.

In [ ]:
style_guide = researcher.resolve_style_guide(
    company_report=company_report,
    product_report=product_report,
    reference_images=reference_images,
    update_mode=update_mode,
)
style_vibe = style_guide.get('vibe', '')
print(f'\n✅ Style guide ready. Vibe: {style_vibe}')


---
## 🖼️ Step 7.6: Image Selection

Only runs if `images` in the receipt is not `No`. This manages the per-product photo library — the actual images that will appear in posts (enhanced with AI or used as inspiration for generation).

These are **separate from style references** — style references inform caption style and tone, while these are the actual visual content of the post.

**Commands:** `add <filepath>` | `select <number>` | `all` | `remove <number>` | Enter when done

After selection, you choose how to use the photos:
- **Option 1:** Enhance the real photo with AI effects
- **Option 2:** Use as visual inspiration for a newly generated image

In [ ]:
image_mode             = receipt.get('images', 'No')
selected_images        = []
enhance_as_inspiration = False  # asked once here, passed to agents via base_brief

if image_mode == 'No':
    print('⏭️  Image mode is No — skipping image selection.')
else:
    company = receipt['company']
    product = receipt['product']
    img_dir = storage.images_dir(company, product)
    img_dir.mkdir(parents=True, exist_ok=True)
    existing = storage.list_images(company, product)

    print(f'\n🖼️  Image library for [{product}]:')
    if existing:
        print(f'  Found {len(existing)} product photo(s):')
        for i, p in enumerate(existing, 1):
            print(f'    {i}. {p.name}')
    else:
        print('  No product photos in library yet.')

    print('\n  Commands: add <filepath> | select <number> | all | remove <number> | Enter')
    print('  Tip: pasting a file path directly also works as add.')
    print()

    while True:
        cmd = input('  > ').strip().strip('"').strip("'")
        if not cmd:
            break

        parts  = cmd.split(None, 1)
        action = parts[0].lower()
        arg    = parts[1].strip() if len(parts) > 1 else ''

        is_path = ('\\' in cmd or '/' in cmd or
                   cmd.lower().endswith(('.png', '.jpg', '.jpeg')))
        if is_path and action not in ('add', 'select', 'remove', 'all'):
            action = 'add'
            arg    = cmd

        if action in ('all', 'select all') or (action == 'select' and arg.lower() == 'all'):
            for img in existing:
                if img not in selected_images:
                    selected_images.append(img)
            print(f'  ✅ Selected all {len(existing)} image(s).')
            break
        elif action == 'add' and arg:
            added = storage.add_image_to_library(company, product, arg)
            if added:
                existing = storage.list_images(company, product)
                if added not in selected_images:
                    selected_images.append(added)
                    print(f'  ✅ Added and selected: {added.name}')
        elif action == 'select' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                chosen = existing[idx]
                if chosen not in selected_images:
                    selected_images.append(chosen)
                    print(f'  ✅ Selected: {chosen.name}')
                else:
                    print(f'  Already selected: {chosen.name}')
            else:
                print(f'  ❌ Invalid number.')
        elif action == 'remove' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                target = existing[idx]
                target.unlink()
                existing = storage.list_images(company, product)
                selected_images = [p for p in selected_images if p != target]
                print(f'  🗑️  Removed: {target.name}')
            else:
                print(f'  ❌ Invalid number.')
        else:
            print('  add <filepath> | select <number> | all | remove <number> | Enter')

    if image_mode == 'Provided Images' and not selected_images:
        if not existing:
            print('\n  ℹ️  No images — falling back to AI Generated.')
            image_mode = 'AI Generated'
            receipt['images'] = 'AI Generated'

    print(f'\n✅ Image mode: {image_mode}')
    print(f'   Selected: {[p.name for p in selected_images] or "none"}')

    # ── Ask enhance vs inspire ONCE here — before parallel execution ─────────
    # If asked inside threads, multiple agents conflict on input().
    if image_mode == 'Provided Images' and selected_images:
        print('\n  How would you like to use these images?')
        print('  1. Enhance real photo with AI effects')
        print('  2. Use as inspiration for a new AI image')
        choice = input('  Enter 1 or 2 (default 1): ').strip()
        enhance_as_inspiration = (choice == '2')


---
## ✍️ Step 8: Generate Posts (Parallel Workflow)

**Agentic Pattern: Parallelization + Evaluator-Optimizer**

This step runs two patterns simultaneously:

**Parallelization:** `InstagramAgent` and `TwitterAgent` run at the same time using `ThreadPoolExecutor`. Twitter (no image) finishes quickly while Instagram generates and audits its image in the background. Platforms are grouped into rounds — "1 Instagram and 2 Twitter" becomes Round 1: [Instagram + Twitter], Round 2: [Twitter].

**A/B Scoring:** Each agent drafts up to 5 caption candidates, scored by `gpt-4o-mini` as a judge. The scorer uses explicit deduction/addition rules — it lists which criteria apply (e.g. `["D1", "D3"]` for deductions, `["A1", "A5"]` for additions) and the score is calculated from the math: start at 8.0, subtract 0.5 per deduction, add 0.5 per addition. This forces real variance instead of clustering at one value.

**Evaluator-Optimizer (image audit):** After a caption is accepted, `gpt-image-2` generates an image and `gpt-4.1` audits it. If it fails, a correction prompt is sent back and the image is regenerated — up to 4 attempts with escalating intensity:
- Attempt 1: Concise actionable fix from the auditor
- Attempt 2: Emotional urgency ("OR I WILL BE FIRED")
- Attempt 3: Nuclear ("NO TEXT AT ALL")
- Attempt 4: Final audit — if still fails, no image is used

"No image is better than a bad image." 

In [ ]:
# ── 1. Build shared context from reports ───────────────────────────────
context = build_context(
    company=company_report,
    product=product_report,
    style=style_guide,
    extra=receipt.get('additional_info', ''),
    month=receipt.get('when', ''),
)

# ── 2. Build agents from per-platform counts ─────────────────────────────
# Build post_platform_list from per-platform counts in the original request.
# e.g. '1 Instagram and 2 Twitter' → [instagram, twitter, twitter]
# This prevents the old cycling bug (instagram, twitter, instagram).
platforms = parse_platforms(receipt.get('platforms', 'instagram'))
n_posts   = int(receipt.get('num_posts', 1))

platform_counts = re.findall(
    r'\b(\d+)\s+(instagram|twitter|blog|x)\b',
    receipt.get('_raw_request', ''), re.I
)
if platform_counts:
    post_platform_list = []
    for count, platform in platform_counts:
        post_platform_list.extend([platform.lower()] * int(count))
else:
    valid_platforms    = [p for p in platforms if p in PLATFORM_AGENTS]
    post_platform_list = [valid_platforms[i % len(valid_platforms)]
                          for i in range(n_posts)]

# Group into parallel rounds
agents_per_post = []
i = 0
while i < len(post_platform_list):
    seen = set()
    round_platforms = []
    j = i
    while j < len(post_platform_list):
        p = post_platform_list[j]
        if p not in seen and p in PLATFORM_AGENTS:
            seen.add(p)
            round_platforms.append(p)
            j += 1
        else:
            break
    if not round_platforms:
        i += 1
        continue
    agents_per_post.append([
        PLATFORM_AGENTS[p](openai_key=OPENAI_API_KEY, review_key=OPENAI_REVIEW_KEY)
        for p in round_platforms
    ])
    i += len(round_platforms)

print(f'🚀 Starting post generation...\n' + '=' * 70)
print(f'📋 Platforms : {", ".join(p.capitalize() for p in post_platform_list)}')
print(f'   Posts     : {len(post_platform_list)} total')
print(f'   Images    : {image_mode}')
print('=' * 70 + '\n')

# ── 3. Build base brief ──────────────────────────────────────────────────
base_brief = {
    'context':                context,
    'total_posts':            len(post_platform_list),
    'image_mode':             image_mode,
    'selected_images':        selected_images,
    'reference_images':       reference_images,
    'style_vibe':             style_vibe,
    'logo_description':       company_report.get('logo_description', '') or '',
    'additional_info':        receipt.get('additional_info', ''),
    'enhance_as_inspiration': enhance_as_inspiration,
    'brand_context': ' | '.join(filter(None, [
        f"Notes: {receipt.get('additional_info','')}" if receipt.get('additional_info') else '',
        f"Vibe: {style_guide.get('vibe','')}" if style_guide.get('vibe') else '',
        f"Tone: {style_guide.get('tone','')}" if style_guide.get('tone') else '',
        f"Product: {product_report.get('product_name','')} — {str(product_report.get('product_description',''))[:80]}" if product_report.get('product_name') else '',
        f"Company: {company_report.get('company_name','')}" if company_report.get('company_name') else '',
    ])),
}


In [ ]:
start_time = time.time()
posts      = run_all_posts(agents_per_post, base_brief)
elapsed    = time.time() - start_time

print(f'\n✨ All agents finished in {elapsed:.2f} seconds')
print('=' * 70)

# ── A/B Score Summary ────────────────────────────────────────────────────
print('\n📊 A/B SCORE SUMMARY')
print('-' * 40)
for i, post in enumerate(posts, 1):
    platform = post.get('platform','?').upper()
    score    = post.get('score', 'N/A')
    audit    = post.get('audit_result', {})
    passed   = audit.get('passed', True)
    reason   = audit.get('reason', '')
    score_str = f'{score:.1f}/10' if isinstance(score, float) else str(score)
    audit_str = '✅ Image passed audit' if passed else f'❌ Image audit failed: {reason}'
    has_image = post.get('image_bytes') is not None
    print(f'  Post {i} [{platform}]  Score: {score_str}  '
          f'{audit_str if has_image else "(no image)"}')
print('-' * 40)

# ── Post content display ─────────────────────────────────────────────────
print('\n📊 GENERATED SOCIAL MEDIA CONTENT')
print('=' * 70 + '\n')

for i, post in enumerate(posts, 1):
    has_image = post.get('image_bytes') is not None
    score     = post.get('score', 'N/A')
    score_str = f'{score:.1f}/10' if isinstance(score, float) else str(score)
    label     = f'Post {i} [{post.get("platform","?").upper()}] — Score: {score_str} {"🖼️" if has_image else ""}'
    print(f'┌─ {label} ' + '─' * max(0, 68 - len(label)))
    print('│')
    for line in post.get('caption', '').split('\n'):
        print(f'│ {line}')
    print('└' + '─' * 69)
    print()

print('=' * 70)
print(f'✅ Generation complete! Total time: {elapsed:.2f}s')
print('=' * 70)


---
## 💾 Step 9: Save Posts & Images

Saves all generated captions, images, and the receipt to the output folder under `AI Storage/{Company}/Created Posts/`. Each post gets its own subfolder with `caption.txt` and `post_image.png` (if an image was generated and passed the audit).

If a post's image failed the audit but you chose to save it anyway, it is saved as `post_image (Failed Audit).png` so it's clearly marked.

In [ ]:
# Save captions and receipt to the output folder
output_dir = storage.save_posts(
    company=receipt['company'],
    product=receipt['product'],
    posts=posts,
    receipt=receipt,
)

# Save any generated or enhanced images into their post subfolder
# image_bytes is None if image_mode was No or generation failed
for i, post in enumerate(posts, 1):
    if post.get('image_bytes'):
        post_dir = output_dir / f'Post {i}'
        storage.save_image(post_dir, post['image_bytes'], filename='post_image.png')

print(f'\n📁 All posts saved to: {output_dir}')

---
## 📅 Step 10: Schedule Posts (Optional)

Only runs if `schedule` is `Yes` in the receipt.

Accepts natural language date input ("2 on June 5th and 1 two days later") and turns it into specific calendar dates using `DATE_PARSE_PROMPT`. Any dates not specified are filled in automatically by GPT and labeled `[auto]`. After you confirm the final schedule, events are pushed directly to Google Calendar via the Google Calendar API.

**Setup required:** Google Calendar credentials JSON from Google Cloud Console. See `INSTRUCTIONS.md` in the Help Documents folder for setup steps.

In [ ]:
if receipt.get('schedule') == 'Yes':
    scheduler = ScheduleAgent(
        openai_key=OPENAI_API_KEY,
        credentials_json=GCAL_CREDENTIALS_JSON,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    scheduler.run(
        receipt=receipt,
        posts=posts,
        output_dir=output_dir,
    )
else:
    print('⏭️  Scheduling skipped.')

---
## 📝 Step 11: Write Log

Writes a combined log file to `AI Storage/{Company}/Log Files/` after every run. The file contains two sections:

1. **Receipt** — what was requested, company/product/style summary, all captions with A/B scores, and output path
2. **Content Audit** — caption audit pass/fail, image audit pass/fail, correction prompts sent on each retry attempt, and overall result

This makes every run fully traceable — you can look back and see exactly what the agent generated, how it scored, and what the guardrails caught.

In [ ]:
logger   = Logger(storage)
log_path = logger.write(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
    posts=posts,
    output_dir=output_dir,
)
print(f'\n🎉 All done! Log: {log_path}')

---
## 🗂️ File Manager — View & Update Core Files

In addition to creating posts, the agent can view and update the three core files saved in `AI Storage/`: **Company Report**, **Product Report**, and **Style Guide**. Log files are view-only.

**Intent Classification:** Before every message in the demo, a small GPT call classifies what the user wants — `create_posts`, `manage`, or `quit` — and routes accordingly. This means you can switch between creating posts and managing files in the same session by just describing what you want in plain English.

The cell below shows how to use the fuzzy match directly to look up an existing report.

In [ ]:
# Example: view a company report directly in the notebook
# Uses the same fuzzy match as the research agent —
# so 'Rita\'s' finds 'Rita\'s Water Ice', 'Rita\'s Watch Ice' finds 'Rita\'s Water Ice' etc.
from difflib import SequenceMatcher

company_input = "Rita's Water Ice"   # ← change this
product_input = "Kiwi Melon"         # ← change this

# Fuzzy match company
existing_companies = storage.list_companies()
input_lower = company_input.lower()
company_match = next(
    (c for c in existing_companies if
     SequenceMatcher(None, input_lower, c.lower()).ratio() > 0.6
     or input_lower in c.lower() or c.lower() in input_lower),
    None
)

if not company_match:
    print(f"No company found matching '{company_input}'. Run Step 4 first.")
else:
    if company_match != company_input:
        print(f"Matched '{company_input}' → [{company_match}]")
    company = company_match

    # Fuzzy match product
    products_dir = storage.products_dir(company)
    existing_products = [
        f.stem.replace(' Product Report', '')
        for f in products_dir.glob('*.json')
    ] if products_dir.exists() else []

    p_lower = product_input.lower()
    product_match = next(
        (p for p in existing_products if
         SequenceMatcher(None, p_lower, p.lower()).ratio() > 0.6
         or p_lower in p.lower() or p.lower() in p_lower),
        product_input
    )
    product = product_match

    # Load and display
    report = storage.load_company_report(company)
    print(f"Company: {report.get('company_name')}")
    print(f"Industry: {report.get('industry')}")
    print(f"Brand voice: {report.get('brand_voice')}")
    print()
    print(json.dumps(report, indent=2))


---
## 🚀 Running the Full Demo

The cells above document how the agent was built step by step. To run the full interactive experience — post creation, file management, and all — use the terminal demo:

```bash
python demo.py
```

The demo mirrors this notebook exactly (Steps 3–11) but wraps it in a polished terminal UI with streaming output, a rich receipt table, and the full intent classification loop so you can switch between creating posts and managing files in one session.

Run the cell below to launch it directly from VS Code, or open a terminal in the project folder and run `python demo.py` manually.

In [2]:
# The interactive demo requires a real terminal — it cannot run inside Jupyter
# because rich UI and input() calls need a proper terminal to work.
#
# To launch the demo, open a terminal in this folder and run:
#
#   python demo.py
#
# Or run the cell below to open a terminal automatically (VS Code only):

import os, subprocess, sys

demo_path = os.path.abspath('demo.py')
folder    = os.path.dirname(demo_path)

if sys.platform == 'win32':
    # Opens a new Command Prompt window and runs demo.py
    subprocess.Popen(
        f'start cmd /k python "{demo_path}"',
        shell=True, cwd=folder
    )
    print(f'✅ Launched demo.py in a new terminal window.')
elif sys.platform == 'darwin':
    # Opens Terminal.app on Mac
    subprocess.Popen(
        ['open', '-a', 'Terminal', demo_path],
        cwd=folder
    )
    print(f'✅ Launched demo.py in Terminal.')
else:
    print('Run this in your terminal:')
    print(f'  python "{demo_path}"')


✅ Launched demo.py in a new terminal window.
